# 3. Exploratory Data Analysis (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
sns.set_theme(style="whitegrid", font_scale=1.2)

## Load data

In [ ]:
from src.data.load_data import load_data

train, test = load_data()

## Process MV

### View MV

In [ ]:
from src.features.missing_values import get_missing_values

print(get_missing_values(train))

In [ ]:
from src.features.missing_values import get_missing_values

print(get_missing_values(test))

In [ ]:
from src.features.missing_values import plot_missing_values

plot_missing_values(train, "Train MV")

In [ ]:
from src.features.missing_values import plot_missing_values

plot_missing_values(test, "Test MV")

### Fill MV

In [ ]:
# Even though train.csv and test.csv have the same structure, pandas can infer different dtypes
# for the same column in different files. This happens because:
# 1. Pandas guesses the dtype based on the first N rows (default 100) of each file.
# 2. Differences in missing values, mixed types, or formatting may lead to different inferred types.
# 3. Newer pandas versions distinguish between object, string[python], and categorical types.
# 
# To ensure consistency and avoid warnings in downstream feature processing,
# we explicitly cast test columns to the same dtype as the corresponding train columns.
for col in train.columns:
    if col in test.columns:
        if pd.api.types.is_integer_dtype(train[col].dtype):
            # use nullable Int64 to support NaN
            test[col] = test[col].astype('Int64')
        else:
            test[col] = test[col].astype(train[col].dtype)

In [ ]:
from src.features.missing_values import process_missing_values
from src.features.missing_values import has_missing_values

train = process_missing_values(train.copy())
print("train have mv? - ", has_missing_values(train))

print()

test = process_missing_values(test.copy())
print("test have mv? - ", has_missing_values(test))

## EDA

### Histogramm SalePrice

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train["SalePrice"], kde=True)
plt.title("SalePrice Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train["LotArea"], kde=True)
plt.title("LotArea Distribution")
plt.show()

### Boxplot for exploring outliers

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=train["GrLivArea"])
plt.title("GrLivArea Boxplot")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=train["LotArea"])
plt.title("LotArea Boxplot")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=train["TotalBsmtSF"])
plt.title("TotalBsmtSF Boxplot")
plt.show()


Show 10 records with the largest GrLivArea

In [ ]:
train.nlargest(10, 'GrLivArea')[['GrLivArea', 'SalePrice']]


There are 2 points that are seems to be not logical:

GrLivArea = 5642, SalePrice = 160000

GrLivArea = 4676, SalePrice = 184750

it's big areas for a very small price. This you can also see on a further scatter plot "GrLivArea vs SalePrice" - 2 points on the right down.

### Scatter-plot: GrLivArea → SalePrice

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=train["GrLivArea"], y=train["SalePrice"])
plt.title("GrLivArea vs SalePrice")
plt.xlabel("GrLivArea")
plt.ylabel("SalePrice")
plt.show()


In [ ]:
plt.figure(figsize=(14, 10))
corr = train.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
top_corr = corr["SalePrice"].abs().sort_values(ascending=False).head(10)
top_corr


We have multicolleniarity for some features. Will left them as is. Will see how it affects different models.

### Search for the important categorical features

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x="OverallQual", y="SalePrice", data=train, estimator="mean")
plt.title("Average SalePrice by OverallQual")
plt.show()


In [ ]:
sns.boxplot(x='OverallQual', y='SalePrice', data=train)
plt.title("SalePrice vs OverallQual")
plt.show()

In [ ]:

f_val, p_val = stats.f_oneway(
    train[train['OverallQual']==1]['SalePrice'],
    train[train['OverallQual']==2]['SalePrice'],
    train[train['OverallQual']==3]['SalePrice'],
    train[train['OverallQual']==4]['SalePrice'],
    train[train['OverallQual']==5]['SalePrice'],
    # train[train['OverallQual']==6]['SalePrice'],
    # train[train['OverallQual']==7]['SalePrice'],
    # train[train['OverallQual']==8]['SalePrice'],
    # train[train['OverallQual']==9]['SalePrice'],
    # train[train['OverallQual']==10]['SalePrice'],

    # ...
)
print(f"F={f_val}, p={p_val}")


## Drop outliers

In [ ]:
from src.features.build_features import drop_outliers

train_clean = drop_outliers(train)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=train_clean["GrLivArea"], y=train_clean["SalePrice"])
plt.title("GrLivArea vs SalePrice")
plt.xlabel("GrLivArea")
plt.ylabel("SalePrice")
plt.show()